# From Batch to Streaming

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/streaming-ml/01-batch-to-streaming

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

**The problem.** Batch ML assumes all data is in memory and you can make many passes. A *stream* is unbounded, arrives one element at a time, and must be processed in a single pass with bounded memory. **The core idea:** summarise recent history with **windows**, and reason in *event time* (when things happened) rather than *processing time* (when you saw them). This shows up in clickstreams, sensor telemetry, fraud, and logs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## 1. From scratch — a stream with event time and processing time

Each event has an **event time** (when it happened) and arrives at a **processing time** = event time + a random network delay. Delays reorder events and make late data the norm.

In [ ]:
N = 300
event_time = np.sort(np.random.randint(0, 100, size=N)).astype(float)
# mostly small delays, with a 5% heavy tail (a phone buffering in a tunnel)
delay = np.random.exponential(1.0, size=N) + (np.random.rand(N) < 0.05) * 8
proc_time = event_time + delay
print('events:', N)
print('max delay:', round(delay.max(), 2), ' | out-of-order fraction:',
      round(np.mean(np.diff(event_time[np.argsort(proc_time)]) < 0), 3))

### Tumbling windows, by hand

A **tumbling window** chops the stream into fixed, non-overlapping blocks and aggregates each. We count events per 10-unit block, keyed by *event* time.

In [ ]:
W = 10
edges = np.arange(0, 110, W)
manual = np.array([np.sum((event_time >= lo) & (event_time < lo + W)) for lo in edges[:-1]])
print('tumbling counts:', manual)

## 2. The library way — validate against pandas

Production stream processors (Flink, Spark) do exactly this windowing. Locally, `pandas` gives the same aggregate — we check our by-hand counts match `groupby` on the event-time bucket.

In [ ]:
import pandas as pd
df = pd.DataFrame({'event_time': event_time})
df['bucket'] = (df['event_time'] // W) * W
lib = df.groupby('bucket').size().reindex(edges[:-1], fill_value=0).to_numpy()
assert np.array_equal(manual, lib), 'by-hand tumbling counts must match pandas groupby'
print('by-hand tumbling window == pandas groupby ✓')

## 3. Visualize it — watermarks and late data

When can we *close* the `[40, 50)` window? If we key by processing time we finalise too early and undercount stragglers. A **watermark** waits until event time (with allowed lateness) has passed.

In [ ]:
lo, hi = 40, 50
in_window = (event_time >= lo) & (event_time < hi)
true_count = int(in_window.sum())
order = np.argsort(proc_time)
seen = np.cumsum(in_window[order])
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(proc_time[order], seen, color='#14b8a6', label='events seen so far')
ax.axhline(true_count, ls='--', color='#eab308', label=f'true count = {true_count}')
ax.set_xlabel('processing time'); ax.set_ylabel('events in [40,50) seen')
ax.set_title('Late arrivals trickle in after the window ends', color='white')
ax.legend(); ax.grid(alpha=0.2); plt.show()

**What to notice:** the count keeps rising *after* processing time 50 — those are late events whose event time is in `[40,50)` but which arrived slowly. Closing the window by processing time at 50 would permanently undercount. A watermark with allowed lateness keeps the window open until event time has demonstrably passed.

## 4. Tradeoffs & when to use it

- **Window types.** *Tumbling* (fixed blocks) for periodic aggregates; *sliding* (overlapping) for smooth recent trends; *landmark* (since a start) for running totals.
- **Event vs processing time.** Always window by **event time** for correctness; processing time is only acceptable when order and latency truly don't matter.
- **Watermark lateness is a latency/completeness knob.** More allowed lateness = more complete windows but higher latency before results finalise.
- **When to stream at all.** If minutes-old data is fine and volume fits memory, batch/micro-batch is simpler. Stream when you need bounded memory over unbounded history or low-latency reactions.

## 5. Your turn

### Exercise 1 — Sliding-window count

Implement a **sliding window** of width `W` that, for a query time `t`, returns the number of events with event time in `(t - W, t]`.

In [ ]:
def sliding_count(event_time, t, W):
    """Count events with (t - W) < event_time <= t."""
    event_time = np.asarray(event_time, dtype=float)
    # TODO(you): boolean-mask the window and sum it
    return ...


In [ ]:
# Checks — run me
ev = np.array([1.0, 5.0, 9.0, 9.5, 20.0])
assert sliding_count(ev, 10, 10) == 4, 'events in (0,10]: 1,5,9,9.5'
assert sliding_count(ev, 9, 5) == 2, 'events in (4,9]: 5 and 9'
assert sliding_count(ev, 100, 10) == 0, 'no events in (90,100]'
assert sliding_count(ev, 20, 0.5) == 1, 'boundary: 20 is included, 19.5 excluded'
print('✅ Exercise 1 passed')

<details>
<summary>💡 Show solution</summary>

```python
def sliding_count(event_time, t, W):
    event_time = np.asarray(event_time, dtype=float)
    return int(np.sum((event_time > t - W) & (event_time <= t)))
```

</details>

### Exercise 2 — A watermark check

Given `allowed_lateness`, a window `[lo, hi)` can be *finalised* once the watermark `max_event_time_seen - allowed_lateness` reaches `hi`. Return the **processing time** at which that first happens (or `None` if it never does).

In [ ]:
def finalize_time(event_time, proc_time, hi, allowed_lateness):
    """Processing time when watermark (max event time seen - lateness) first reaches hi."""
    order = np.argsort(proc_time)
    # TODO(you): walk events in processing-time order, track max event time seen,
    #            return the proc_time when (max_seen - allowed_lateness) >= hi
    return ...


In [ ]:
# Checks — run me
et = np.array([5.0, 45.0, 48.0, 60.0])
pt = np.array([5.1, 45.2, 49.0, 61.0])
# watermark reaches 50 once we see event time >= 50 + lateness(3) = 53 -> the 60.0 event at pt 61.0
assert finalize_time(et, pt, hi=50, allowed_lateness=3) == 61.0
assert finalize_time(et, pt, hi=50, allowed_lateness=0) == 61.0
assert finalize_time(np.array([1.0]), np.array([1.0]), hi=50, allowed_lateness=0) is None
print('✅ Exercise 2 passed')

<details>
<summary>💡 Show solution</summary>

```python
def finalize_time(event_time, proc_time, hi, allowed_lateness):
    order = np.argsort(proc_time)
    max_seen = -np.inf
    for i in order:
        max_seen = max(max_seen, event_time[i])
        if max_seen - allowed_lateness >= hi:
            return float(proc_time[i])
    return None
```

</details>

## 6. Key takeaways

- A stream is **unbounded, one-pass, bounded-memory**; **windows** make it finite.
- Distinguish **event time** from **processing time**; window by event time.
- **Watermarks** trade latency for completeness when finalising windows over late, out-of-order data.
- Next: [Streaming Algorithms & Sketches](https://ml-viz-ruby.vercel.app/courses/streaming-ml/02-streaming-algorithms) — sublinear-memory summaries of the stream.